# Data Collection and Source Gathering

This notebook gathers the study data from the selected source layers used in this project. It answers a simple but important question: where did the text, prices, and event data actually come from, and why were those sources selected?

The answer is mixed:

- SEC filing text is scraped from EDGAR.
- Market price history is downloaded from Yahoo Finance.
- Official-event and statement-event timelines are curated row lists stored in code.
- GDELT topic features can add broader external news context when enabled.

Those source layers were selected because they map directly to the research question:

- filings provide the raw text that can contain supply-risk language
- market prices provide the return, volatility, beta, and oil-context features
- official and statement events provide the dated geopolitical event layer
- optional web-news enrichment can add broader external text context when explicitly enabled

All data-gathering logic runs from the local `src/oil_projects` package inside this study folder.

## Source Map

This section makes the acquisition method explicit so the study is not ambiguous about what was scraped, what was fetched through an API, and what was manually curated.

In [ ]:
from __future__ import annotations

import sys
from pathlib import Path


def find_project_dir(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / 'src' / 'oil_projects').exists() and (candidate / 'README.md').exists():
            return candidate
    raise FileNotFoundError('Run this notebook from inside the war_sentiment_study folder.')


# Point the notebook at the study folder and package copy that travel with this notebook.
PROJECT_DIR = find_project_dir(Path.cwd().resolve())
SRC_DIR = PROJECT_DIR / 'src'
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

import importlib
import importlib.util
import subprocess

def ensure_parquet_engine() -> None:
    if importlib.util.find_spec('pyarrow') is not None or importlib.util.find_spec('fastparquet') is not None:
        return
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'pyarrow', '-q'])
    importlib.invalidate_caches()

ensure_parquet_engine()

import pandas as pd
try:
    from IPython.display import display
except Exception:
    def display(obj):
        print(obj)

# Remap the project config so any pipeline code writes into this study folder.
import oil_projects.config as _config
import oil_projects.common as _common

_config.BASE_DIR = PROJECT_DIR
_config.DATA_DIR = PROJECT_DIR / 'data'
_config.ARTIFACT_DIR = PROJECT_DIR / 'artifacts'
_config.RAW_DIR = _config.DATA_DIR / 'raw'
_config.PROCESSED_DIR = _config.DATA_DIR / 'processed'

BASE_DIR = _config.BASE_DIR
DATA_DIR = _config.DATA_DIR
ARTIFACT_DIR = _config.ARTIFACT_DIR
RAW_DIR = _config.RAW_DIR
PROCESSED_DIR = _config.PROCESSED_DIR

_common.BASE_DIR = BASE_DIR
_common.DATA_DIR = DATA_DIR
_common.ARTIFACT_DIR = ARTIFACT_DIR
_common.RAW_DIR = RAW_DIR
_common.PROCESSED_DIR = PROCESSED_DIR

from oil_projects.news_pipeline import _download_target_prices, _fetch_filings, _build_daily_features
from oil_projects.quant_pipeline import _build_feature_panel
from oil_projects.official_sources import load_official_event_frame, build_official_event_features
from oil_projects.statement_sources import load_statement_event_frame, build_statement_event_features
from oil_projects.config import USE_WEB_NEWS_TOOLS

# Make sure the expected output folders exist before the build starts.
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)


In [ ]:
source_map = pd.DataFrame(
    [
        {
            'component': 'SEC filing text',
            'method': 'Web scrape',
            'source': 'SEC EDGAR primary filing documents',
            'implementation': 'requests + BeautifulSoup in news_pipeline._fetch_filings()',
            'output': 'data/processed/raw_news.parquet',
        },
        {
            'component': 'Daily market prices',
            'method': 'API download',
            'source': 'Yahoo Finance',
            'implementation': 'yfinance in news_pipeline._download_target_prices() and quant_pipeline._build_feature_panel()',
            'output': 'data/processed/news_panel.parquet and data/processed/quant_panel.parquet',
        },
        {
            'component': 'FinBERT sentiment cache',
            'method': 'Local model inference',
            'source': 'ProsusAI/finbert with VADER fallback',
            'implementation': 'news_pipeline._finbert_scores()',
            'output': 'data/processed/finbert_scores.json',
        },
        {
            'component': 'Official event timeline',
            'method': 'Curated rows',
            'source': 'White House, Treasury, OPEC references stored in code',
            'implementation': 'official_sources.py',
            'output': 'artifacts/official_event_sources.csv',
        },
        {
            'component': 'Statement event timeline',
            'method': 'Curated rows',
            'source': 'Selected dated war/shipping/sanctions events stored in code',
            'implementation': 'statement_sources.py',
            'output': 'artifacts/statement_event_sources.csv',
        },
        {
            'component': 'Broader web-news topics',
            'method': 'API download (optional)',
            'source': 'GDELT',
            'implementation': 'requests in web_news_tools.fetch_gdelt_topic_features()',
            'output': 'data/processed/gdelt_topic_features.parquet',
        },
    ]
)

source_map


## Data Gathering Process

This cell gathers the selected source data and transforms it into the local datasets used by the training and analysis notebooks:

- `raw_news.parquet`
- `news_panel.parquet`
- `quant_panel.parquet`
- `finbert_scores.json`
- `gdelt_topic_features.parquet` when web-news enrichment is enabled
- official and statement event source files

It does not train the models. It only gathers the source data and turns it into the local inputs used later in the study.

In [ ]:
# Download the local price history used by both the news and quant feature builders.
target_prices = _download_target_prices()

# Gather the raw filing-level text records from EDGAR and cache them locally.
filings = _fetch_filings()
filings.to_parquet(PROCESSED_DIR / 'raw_news.parquet', index=False)

# Build the daily text-and-event panel used by the sent side of the study.
news_panel = _build_daily_features(filings, target_prices)
news_panel.to_parquet(PROCESSED_DIR / 'news_panel.parquet', index=False)

# Build the daily market-and-structure panel used by the baseline backbone.
quant_panel = _build_feature_panel()
quant_panel.to_parquet(PROCESSED_DIR / 'quant_panel.parquet', index=False)

# Materialize the dated official and statement event layers that feed the event-context features.
official_event_features = build_official_event_features()
statement_event_features = build_statement_event_features()

# Summarize the local outputs so the notebook shows what was built in this run.
summary = pd.DataFrame(
    [
        {'artifact': 'raw_news.parquet', 'rows': len(filings), 'path': str(PROCESSED_DIR / 'raw_news.parquet')},
        {'artifact': 'news_panel.parquet', 'rows': len(news_panel), 'path': str(PROCESSED_DIR / 'news_panel.parquet')},
        {'artifact': 'quant_panel.parquet', 'rows': len(quant_panel), 'path': str(PROCESSED_DIR / 'quant_panel.parquet')},
        {'artifact': 'official_event_sources.csv', 'rows': len(load_official_event_frame()), 'path': str(ARTIFACT_DIR / 'official_event_sources.csv')},
        {'artifact': 'statement_event_sources.csv', 'rows': len(load_statement_event_frame()), 'path': str(ARTIFACT_DIR / 'statement_event_sources.csv')},
    ]
)

if USE_WEB_NEWS_TOOLS:
    gdelt_path = PROCESSED_DIR / 'gdelt_topic_features.parquet'
    if gdelt_path.exists():
        gdelt_frame = pd.read_parquet(gdelt_path)
        summary.loc[len(summary)] = {
            'artifact': 'gdelt_topic_features.parquet',
            'rows': len(gdelt_frame),
            'path': str(gdelt_path),
        }

summary


## Output Check

This final section shows a small preview of the gathered datasets so you can confirm that the scrape/download/transform pipeline completed as expected.

In [ ]:
# Show a quick sanity check of the built panels and event files.
print('GDELT web-news features enabled:', USE_WEB_NEWS_TOOLS)

print('\nRaw filings preview')
display(filings[['ticker', 'form', 'filing_date', 'accepted_at']].head())

print('\nNews panel preview')
display(news_panel[['date', 'ticker', 'target', 'filing_count', 'avg_sentiment', 'official_event_weight']].head())

print('\nQuant panel preview')
display(quant_panel[['date', 'ticker', 'target', 'ret_1', 'vol_10', 'official_event_weight']].head())

print('\nOfficial events preview')
display(load_official_event_frame())

print('\nStatement events preview')
display(load_statement_event_frame())
